Embed every turn as a vector and retrieve the top-K most relevant past turns at inference time. This brings semantic search to conversation memory.

Think of a library with no catalog. To find anything, you'd walk shelf by shelf. Now imagine a librarian who instantly knows which three books best answer your question. Vector Store Memory gives your agent that librarian. Instead of finding context by when it was said, it finds context by what it means.

In the previous short-term memory notebooks, we explored recency-based strategies: keep the last k messages (sliding window) or compress old turns into a summary. Both rely on when something was said, not what was said.

Vector Store Memory takes a different approach. It converts every conversation turn into a dense embedding vector (a list of numbers capturing meaning). It stores that vector in a vector database. When the agent needs context, the current query is embedded and compared against all stored turns using cosine similarity. The top-K most semantically relevant fragments come back, regardless of when they occurred.

This is the same idea behind RAG (Retrieval-Augmented Generation, where you fetch relevant text before generating a response). Here we apply it to the agent's own conversation history instead of external documents.

The payoff: an agent that can recall a fact from turn 3 when the user asks about it at turn 47. A sliding window would have dropped that fact long ago.

By the end of this notebook you'll understand:

How to build a vector store memory from scratch using OpenAI embeddings and ChromaDB.

The retrieval pipeline: embed, store, query, inject into prompt.

A controlled 50-turn experiment proving that semantic recall dramatically outperforms sliding window for non-sequential questions.

The tradeoffs: retrieval quality, cost, latency, and when to combine this with other memory strategies.

#### Key Concepts

Embedding model: A model (e.g., OpenAI text-embedding-3-small) that maps text to a fixed-length vector capturing its meaning. Similar text produces similar vectors.

Vector database: A store optimized for approximate nearest-neighbor (ANN) search over high-dimensional vectors. ANN means "find the closest matches fast, even among millions." We use ChromaDB (in-memory) here.

Cosine similarity: The distance metric we use to compare vectors. Higher cosine similarity means more semantically related.

Top-K retrieval: Return the K most similar stored vectors for a given query. K controls the recall/cost tradeoff. Low K is cheap but might miss things. High K catches more but costs more tokens.

Chunk strategy: How you segment conversations for embedding: individual messages, user-assistant pairs, or sliding windows. We embed user-assistant pairs for coherent retrieval.

Metadata: Timestamps, turn numbers, and other attributes attached to each stored vector. These enable hybrid filtering (e.g., "relevant and recent").

Context injection: Retrieved memories are formatted and prepended to the prompt so the LLM can reference them.

> **The architectural shift this notebook represents:**
>
> | Techniques 01–05 | Technique 06+ |
> |---|---|
> | Memory lives in RAM | Memory lives in a **database** |
> | Memory resets when the session ends | Memory **survives** across sessions |
> | Context retrieved by **position** (recency) | Context retrieved by **semantic similarity** |
> | One user, one buffer | **Multi-tenant** — thousands of users, one store |
> | Storage: Python list | Storage: **vector database** (ChromaDB) |
>
> Everything changes here. This is long-term memory.

---

## What Is Vector Store Memory?

### The core idea in one sentence
Convert every conversation turn into a vector (a numerical representation of meaning), store it in a persistent database, and at the start of every new turn, retrieve the most **semantically relevant** past messages — not just the most recent ones.

---

### The mental model — a searchable filing cabinet

Techniques 01–05 were like a notepad on the advisor's desk. You could only read what was written recently. When the notepad was full, old pages fell off.

Vector Store Memory is like a **searchable filing cabinet** that remembers everything. Every conversation turn — from every session, ever — is filed away. When the user asks a new question, the system searches the cabinet for the most relevant past conversations and brings those files to the desk. The advisor reads those files alongside the current conversation and gives a personalised answer.

```
User asks: "Should I rebalance my portfolio?"
                    ↓
    Embed the query into a vector
                    ↓
    Search ChromaDB for similar past messages
                    ↓
    Retrieved: 
      - "I'm conservative with investments" (3 months ago)
      - "I have ₹2L in equity mutual funds" (2 sessions ago)
      - "I'm planning to buy a house in 2 years" (last session)
                    ↓
    Inject retrieved context + current conversation into API call
                    ↓
    FinCoach gives personalised, cross-session advice
```

---

### Point 1 — What is an embedding?

An embedding is a list of numbers (a vector) that represents the **meaning** of a piece of text. Sentences with similar meaning have vectors that are close together in space. Sentences with different meaning have vectors that are far apart.

```
"My salary is ₹1,20,000"    → [0.12, -0.45, 0.78, ..., 0.33]  (1536 numbers)
"I earn ₹1.2 lakh per month" → [0.13, -0.44, 0.77, ..., 0.34]  ← SIMILAR VECTOR
"I enjoy playing cricket"    → [0.89,  0.21, -0.45, ..., 0.12]  ← DIFFERENT VECTOR
```

When you search with a query like "what is my income", its embedding is close to both salary messages — even though they use different words. This is **semantic search** — finding meaning, not just keywords.

We use OpenAI's `text-embedding-3-small` model to generate embeddings. It produces 1536-dimensional vectors at $0.02 per million tokens — very cheap.

---

### Point 2 — What is ChromaDB?

ChromaDB is an open-source vector database. It stores embeddings alongside the original text and metadata, and lets you search by vector similarity. For development and small-scale production, it runs entirely in-process with no server required. For large-scale production, it has a server mode and managed cloud options.

We use ChromaDB because:
- Zero infrastructure — runs in-process, data persisted to disk
- Simple Python API — easy to understand the mechanics
- Production-credible — used in real production systems
- Free and open source

Production alternatives: pgvector (PostgreSQL-native), Pinecone (managed), Qdrant (high-scale), Weaviate.

---

### Point 3 — The retrieval-augmented memory loop

The memory loop for Vector Store Memory has two extra steps compared to buffer memory:

```
WRITE (at the end of each turn):
  1. Embed the user message
  2. Store (embedding + text + metadata) in ChromaDB

READ (at the start of each turn):
  1. Embed the current user query
  2. Search ChromaDB for the k most similar past messages
  3. Inject retrieved messages as context into the API call
```

This is **Retrieval-Augmented Memory** — the same idea as RAG, but applied to conversation history instead of documents.

---

### Point 4 — What metadata to store alongside the embedding

Every entry in ChromaDB carries metadata — structured fields that enable filtering. For FinCoach, we store:

```python
metadata = {
    "user_id":    "chiru_001",    # Multi-tenant isolation — CRITICAL
    "session_id": "sess_abc123",  # Which session this came from
    "role":       "user",         # Was this the user or the assistant?
    "timestamp":  "2025-06-12T...",# When was this said?
    "turn":       3,              # Which turn number?
}
```

`user_id` in metadata is the most critical production field. Without it, User A's financial history is retrievable when serving User B. This is a privacy and regulatory violation in financial services.

---

### Point 5 — The filing cabinet problem: it has no concept of time

Vector Store Memory retrieves by **semantic similarity**, not recency. A message from three years ago that is semantically similar to the current query will be retrieved — even if the situation has completely changed.

Example: User said "I work at Infosys" in 2022. They changed jobs in 2025 and said "I now work at TCS." Both messages are semantically similar to the query "where do I work." The vector store returns both — and the model may blend them or pick the wrong one.

This is the **stale fact problem** — the fundamental limitation of pure vector store memory. It is why Technique 08 (Knowledge Graph) and Technique 24 (Graphiti) exist. For now, we mitigate it with:
- Timestamp metadata + recency weighting in retrieval
- Limiting retrieval to recent sessions only

---

## Trade-offs

| | |
|---|---|
| ✅ Cross-session persistence — memory survives restarts | ❌ No concept of time — stale facts can surface |
| ✅ Semantic retrieval — finds relevant context by meaning | ❌ Retrieval quality depends on embedding model quality |
| ✅ Multi-tenant by design (metadata filtering) | ❌ Extra embedding call per turn (cost + latency) |
| ✅ Scales to millions of messages | ❌ No relationship between facts (who owns what) |
| ✅ Works across all sessions, all time | ❌ Cannot answer "what is current" vs "what was historical" |

## Production Verdict

> **The production standard for long-term memory. Use it. But know its limits.**
>
> Vector Store Memory is the most widely deployed long-term memory pattern in production agent systems today. It handles cross-session persistence, semantic retrieval, and multi-tenancy elegantly. Its limitation — the stale fact problem — becomes serious in domains where facts change over time (financial status, employment, health). The fix is Graphiti (Technique 24) — a temporal knowledge graph that tracks when facts were true. For now, we implement vector store memory correctly and we call out exactly where it will fail.

---

In [1]:
# Install required packages.
# 'openai'      — OpenAI SDK: GPT-4o for conversation, text-embedding-3-small for embeddings.
# 'chromadb'    — Vector database for storing and searching embeddings.
# 'tiktoken'    — Exact GPT-4o tokeniser for token counting.
!pip install openai chromadb tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # Session persistence.
import os                                # Environment variables.
import time                              # Rate-limit delays.
import uuid                              # Unique IDs for ChromaDB documents.
from datetime import datetime, timezone  # UTC timestamps.
from typing import List, Dict, Optional  # Type hints.
from dataclasses import dataclass, field, asdict  # Data models.

# --- Third-party ---
import openai    # OpenAI SDK — used for both chat AND embeddings.
import chromadb  # Vector database.
import tiktoken  # Exact GPT-4o tokeniser.

In [3]:
# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets."

# --- Model constants ---
client = openai.OpenAI(api_key=OPENAI_API_KEY)
# Single client for BOTH chat completions AND embeddings.

CHAT_MODEL      = "gpt-4o"
# Main conversation model — same as all previous techniques.

EMBEDDING_MODEL = "text-embedding-3-small"
# OpenAI's embedding model — converts text to 1536-dimensional vectors.
# Pricing: $0.02 per million tokens — very cheap for production use.
# Alternative: text-embedding-3-large (3072-dim, higher quality, 5x more expensive).
# For most FinCoach use cases, small is sufficient.

EMBEDDING_DIMENSIONS = 1536
# The vector size produced by text-embedding-3-small.
# Every document in ChromaDB is stored as a 1536-element float list.
# ChromaDB must be configured with this dimension to accept our vectors.

TOKENISER = tiktoken.get_encoding("o200k_base")
# Exact tokeniser for gpt-4o.

# --- ChromaDB setup ---
CHROMA_PERSIST_DIR = "/tmp/fincoach_chromadb"
# Directory where ChromaDB persists data to disk.
# Data survives Python process restarts — this is our cross-session storage.
# In production: use a persistent volume or managed ChromaDB server.

chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
# PersistentClient: data is written to disk and survives between runs.
# Alternative: chromadb.Client() for in-memory only (no persistence).
# Alternative: chromadb.HttpClient() for a remote ChromaDB server.

COLLECTION_NAME = "fincoach_memory"
# Name of the ChromaDB collection — like a table in a relational database.
# All users' memories go into one collection, separated by user_id metadata.

print(f"Chat model      : {CHAT_MODEL}")
print(f"Embedding model : {EMBEDDING_MODEL} ({EMBEDDING_DIMENSIONS}D vectors)")
print(f"Vector store    : ChromaDB at {CHROMA_PERSIST_DIR}")

Key loaded from environment variable.
Chat model      : gpt-4o
Embedding model : text-embedding-3-small (1536D vectors)
Vector store    : ChromaDB at /tmp/fincoach_chromadb


---
## Part 1 — The Message Data Model

Same as previous techniques, with one addition: a `memory_id` field for the ChromaDB document ID.

In [4]:
@dataclass
class Message:
    """A single conversation message — extended for vector store storage."""

    role: str
    # 'user' or 'assistant'.

    content: str
    # The message text.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set at creation.

    memory_id: str = field(
        default_factory=lambda: str(uuid.uuid4())
    )
    # Unique ID for this message in ChromaDB.
    # Generated once when the message is created — used as the ChromaDB document ID.
    # uuid.uuid4() generates a random UUID like '3f2504e0-4f89-11d3-9a0c-0305e82c3301'.
    # In production: this is also your audit trail key — link the ChromaDB record
    # back to your relational database via this ID.

    def to_api_format(self) -> Dict[str, str]:
        """Return only role and content — what the OpenAI API accepts."""
        return {"role": self.role, "content": self.content}

    def token_count(self) -> int:
        """Token count using the exact GPT-4o tokeniser."""
        return len(TOKENISER.encode(self.content))

print("Message dataclass defined.")

Message dataclass defined.


---
## Part 2 — The Embedding Engine

The embedding engine is a standalone component — it converts text to vectors and back. Built separately so it is independently testable and swappable.

In [5]:
def embed_text(text: str) -> List[float]:
    """
    Convert a text string into a 1536-dimensional embedding vector.
    Uses OpenAI's text-embedding-3-small model.

    Args:
        text: The text to embed — any length, any language.

    Returns:
        A list of 1536 floats — the semantic representation of the text.

    Production notes:
    - Cache embeddings for identical strings — they are deterministic.
    - Batch embed multiple texts in one API call for efficiency.
    - text-embedding-3-small has a maximum input of 8191 tokens.
      Truncate long messages before embedding.
    """

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        # text-embedding-3-small: 1536-dim, cheap, production-grade quality.

        input=text,
        # The text to embed.
        # OpenAI also accepts a list of strings for batch embedding:
        # input=[text1, text2, text3] returns embeddings for all three in one call.
        # Batch embedding is ~3x cheaper per token in practice.
    )

    return response.data[0].embedding
    # response.data is a list of embedding objects.
    # [0] gets the first (and for single-text input, only) embedding.
    # .embedding is the list of 1536 floats.


def embed_batch(texts: List[str]) -> List[List[float]]:
    """
    Embed multiple texts in a single API call — more efficient than looping.
    Use this when storing multiple messages at once (e.g. end-of-session batch write).

    Args:
        texts: A list of strings to embed.

    Returns:
        A list of embedding vectors, one per input text, in the same order.
    """

    if not texts:
        return []
        # Guard against empty input — would cause an API error.

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
        # Pass the full list — OpenAI embeds all in one API call.
        # Maximum batch size: 2048 texts per call.
    )

    # Sort by index to ensure the output order matches the input order.
    response.data.sort(key=lambda x: x.index)
    # The API may return embeddings in any order — sorting by index is required
    # to ensure texts[i] corresponds to returned_embeddings[i].

    return [item.embedding for item in response.data]
    # Return a list of embedding lists — one per input text.


# --- Quick sanity check ---
test_embedding = embed_text("My monthly salary is ₹1,20,000")
print(f"Embedding generated: {len(test_embedding)} dimensions")
print(f"First 5 values: {[round(v, 4) for v in test_embedding[:5]]}")
print(f"This vector represents the meaning of 'My monthly salary is ₹1,20,000'")

Embedding generated: 1536 dimensions
First 5 values: [-0.0197, 0.0008, 0.0511, -0.0057, -0.0332]
This vector represents the meaning of 'My monthly salary is ₹1,20,000'


---
## Part 3 — The VectorStoreMemory Class

This class manages two zones:
1. **ChromaDB** — the persistent long-term store (all messages, all sessions, all time)
2. **Recent buffer** — the current session's recent turns (same as Technique 05)

On every turn: recent buffer provides recency, ChromaDB provides long-term relevance.

In [6]:
class VectorStoreMemory:
    """
    Long-term memory using ChromaDB for persistent, semantic retrieval.

    Architecture:
    ┌──────────────────────────────────────────────────────────────┐
    │  ChromaDB (persistent, cross-session)                         │
    │  • Every user message embedded and stored                     │
    │  • Filtered by user_id — multi-tenant safe                    │
    │  • Retrieved by semantic similarity at query time             │
    │  • Survives service restarts, session boundaries              │
    ├──────────────────────────────────────────────────────────────┤
    │  Recent buffer (in-memory, current session only)              │
    │  • Last max_recent_turns turns verbatim                       │
    │  • Provides conversational coherence for the current session  │
    └──────────────────────────────────────────────────────────────┘

    API call context = [system] + [retrieved memories] + [recent buffer]
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_recent_turns: int = 5,
        # Number of recent turns to keep verbatim in the buffer.
        # These provide conversational flow for the current session.
        # The vector store handles everything older than this.
        retrieval_k: int = 3,
        # How many past messages to retrieve from ChromaDB per turn.
        # k=3: retrieve the 3 most semantically relevant past messages.
        # Tune this: higher k = more context but more tokens per call.
        # Production guide: start at 3, increase if recall is missing,
        # decrease if context is noisy.
        store_assistant_messages: bool = False,
        # Whether to embed and store assistant messages in ChromaDB.
        # Default False: only store user messages.
        # Reasoning: user messages contain the critical facts (salary, goals, risk).
        # Assistant messages are advice — less useful to retrieve later.
        # Setting True roughly doubles the embedding call count and storage.
    ):
        self.session_id   = session_id
        self.user_id      = user_id
        self.system_prompt = system_prompt
        self.max_recent_turns     = max_recent_turns
        self.retrieval_k          = retrieval_k
        self.store_assistant_messages = store_assistant_messages

        # --- ChromaDB collection ---
        self._collection = chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            # get_or_create: if the collection exists, open it; if not, create it.
            # This is idempotent — safe to call multiple times.

            metadata={"hnsw:space": "cosine"},
            # Use cosine similarity for distance calculation.
            # Cosine measures the angle between vectors — not their magnitude.
            # This is the standard for text embeddings because the length of the
            # text should not affect similarity — only the direction (meaning) matters.
            # Alternative: 'l2' (Euclidean distance) — less appropriate for text.
        )

        # --- Recent buffer ---
        self._recent_buffer: List[Message] = []
        # The current session's recent turns — verbatim, in context every call.
        # This is the same pattern as Technique 05, but bounded by max_recent_turns
        # rather than a token count. For simplicity in this technique.

        # --- Session stats ---
        self._total_turns     = 0
        self._stored_count    = 0   # Messages embedded and stored in ChromaDB.
        self._retrieved_count = 0   # Total messages retrieved across all turns.
        self.created_at = datetime.now(timezone.utc).isoformat()

        existing = self._collection.count()
        # Count total documents in the collection — across ALL users.

        print(f"[VectorStore] Initialised — session: {self.session_id}")
        print(f"  User ID             : {self.user_id}")
        print(f"  Recent buffer turns : {self.max_recent_turns}")
        print(f"  Retrieval k         : {self.retrieval_k} messages per turn")
        print(f"  ChromaDB collection : '{COLLECTION_NAME}' ({existing} existing docs)")

    # ------------------------------------------------------------------
    # WRITE — Embed and store a message
    # ------------------------------------------------------------------

    def _store_in_vector_db(self, message: Message) -> None:
        """
        Embed a message and store it in ChromaDB with metadata.
        This is the WRITE half of the memory loop.
        Called after every user message (and optionally assistant messages).
        """

        embedding = embed_text(message.content)
        # Convert the message text to a 1536-dimensional vector.
        # This is the embedding call — one API call per stored message.
        # In production: batch these at session end rather than per-turn
        # to reduce API round-trips.

        self._collection.add(
            ids=[message.memory_id],
            # Unique document ID — the uuid4 generated in the Message dataclass.
            # ChromaDB uses this to deduplicate — same ID = update, not duplicate.

            embeddings=[embedding],
            # The vector representation of this message.
            # Wrapped in a list because ChromaDB accepts batch adds.

            documents=[message.content],
            # The original text — stored alongside the embedding.
            # Retrieved alongside the embedding during search.

            metadatas=[{
                "user_id":    self.user_id,
                # CRITICAL for multi-tenancy — every query filters by this.
                # Without this, User A's memories are visible to User B.

                "session_id": self.session_id,
                # Which session this message came from.
                # Useful for session-scoped queries or debugging.

                "role":       message.role,
                # 'user' or 'assistant' — allows filtering by role.

                "timestamp":  message.timestamp,
                # When this was said — for recency weighting and temporal filtering.
                # In ChromaDB: metadata values must be str, int, float, or bool.
                # Timestamps as ISO strings are fine for filtering.

                "turn":       self._total_turns,
                # Turn number — useful for ordering results.
            }]
        )

        self._stored_count += 1
        # Track how many messages have been stored in ChromaDB this session.

    # ------------------------------------------------------------------
    # READ — Retrieve relevant memories
    # ------------------------------------------------------------------

    def _retrieve_memories(
        self,
        query: str,
        exclude_current_session: bool = False
    ) -> List[Dict]:
        """
        Search ChromaDB for the k most relevant past messages for this user.
        This is the READ half of the memory loop.
        Called at the start of every turn, before the API call.

        Args:
            query:                    The current user message — used as the search query.
            exclude_current_session:  If True, only return memories from past sessions.
                                      Useful to avoid duplicating recent buffer content.

        Returns:
            List of dicts, each with 'content', 'metadata', and 'distance' keys.
        """

        if self._collection.count() == 0:
            # ChromaDB is empty — no memories to retrieve.
            # This is normal on the very first turn of the very first session.
            return []

        query_embedding = embed_text(query)
        # Embed the current query to get a search vector.
        # ChromaDB will compare this vector against all stored vectors
        # and return the most similar ones.

        # Build the metadata filter — ALWAYS filter by user_id.
        where_filter = {"user_id": {"$eq": self.user_id}}
        # $eq: exact match. This filters to ONLY this user's memories.
        # Without this filter, the search spans all users — a privacy violation.

        if exclude_current_session:
            where_filter = {
                "$and": [
                    {"user_id":    {"$eq": self.user_id}},
                    {"session_id": {"$ne": self.session_id}},
                    # $ne: not equal — exclude current session.
                    # Returns only memories from previous sessions.
                ]
            }

        try:
            results = self._collection.query(
                query_embeddings=[query_embedding],
                # The search vector — ChromaDB compares this against all stored embeddings.

                n_results=min(self.retrieval_k, self._get_user_memory_count()),
                # How many results to return.
                # min() prevents requesting more results than exist — would cause an error.

                where=where_filter,
                # Metadata filter — scopes the search to this user.

                include=["documents", "metadatas", "distances"],
                # What to include in results:
                # documents: the original text strings
                # metadatas: the metadata dicts we stored
                # distances: the cosine distance score (lower = more similar)
            )
        except Exception as e:
            # Defensive: if retrieval fails (e.g. no documents match the filter),
            # return empty rather than crashing the agent.
            print(f"  [RETRIEVE] Warning: retrieval failed: {e}")
            return []

        # Reshape the ChromaDB results into a cleaner format.
        memories = []
        if results["documents"] and results["documents"][0]:
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            ):
                # results["documents"] is a list of lists — [0] gets the first query's results.
                # We only have one query per call, so [0] always applies.
                memories.append({
                    "content":  doc,
                    "metadata": meta,
                    "distance": round(dist, 4),
                    # Distance: 0.0 = identical meaning, 1.0 = completely different.
                    # For cosine distance, values under 0.3 are typically very relevant.
                })

        self._retrieved_count += len(memories)
        return memories

    def _get_user_memory_count(self) -> int:
        """Count how many memories exist for this specific user in ChromaDB."""
        try:
            results = self._collection.get(
                where={"user_id": {"$eq": self.user_id}}
            )
            return len(results["ids"])
        except Exception:
            return 0

    # ------------------------------------------------------------------
    # CORE OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Add a message to the recent buffer AND to ChromaDB.
        The ChromaDB write makes this message retrievable in all future sessions.
        """

        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'. Use 'user' or 'assistant'.")

        new_message = Message(role=role, content=content)

        # --- Write to recent buffer ---
        self._recent_buffer.append(new_message)
        # Add to the in-memory recent buffer for conversational coherence.

        if len(self._recent_buffer) > self.max_recent_turns * 2:
            # Evict the oldest message if the buffer exceeds its size limit.
            # max_recent_turns * 2 because each turn = 2 messages (user + assistant).
            self._recent_buffer.pop(0)

        # --- Write to ChromaDB (long-term) ---
        should_store = (
            role == "user" or
            (role == "assistant" and self.store_assistant_messages)
        )
        # Only store user messages by default.
        # Storing assistant messages doubles the storage and embedding cost.
        # The retrieval benefit is modest — user messages carry the key facts.

        if should_store:
            self._store_in_vector_db(new_message)
            # Embed and persist to ChromaDB.

        if role == "assistant":
            self._total_turns += 1

    def get_messages_for_api(
        self,
        current_query: str,
        similarity_threshold: float = 0.7
    ) -> List[Dict[str, str]]:
        """
        Build the complete message list for the OpenAI API.

        Structure:
          [system: FinCoach persona]
          [system: retrieved memories from past sessions]  ← semantic match
          [recent buffer messages verbatim]               ← current session recency

        Args:
            current_query:        The user's current message — used for retrieval search.
            similarity_threshold: Only include retrieved memories with distance < this.
                                  Filters out marginally-relevant memories.
                                  Lower threshold = stricter filtering.
        """

        messages = []

        # 1. System prompt — FinCoach's persona.
        messages.append({"role": "system", "content": self.system_prompt})

        # 2. Retrieved memories from ChromaDB.
        retrieved = self._retrieve_memories(
            query=current_query,
            exclude_current_session=True,
            # exclude_current_session=True: only retrieve from PAST sessions.
            # The current session's recent turns are in the buffer below — no duplication.
        )

        # Filter by similarity threshold — only inject highly relevant memories.
        relevant = [m for m in retrieved if m["distance"] < similarity_threshold]
        # distance < 0.7 means the memory is reasonably similar to the query.
        # Lower threshold = stricter = fewer, more relevant memories.
        # Tune this based on your domain — financial advice benefits from precision.

        if relevant:
            # Format retrieved memories as a single system context block.
            memory_lines = []
            for mem in relevant:
                ts = mem["metadata"].get("timestamp", "")[:10]  # Date only.
                memory_lines.append(
                    f"[{ts}] {mem['metadata'].get('role', 'user')}: {mem['content']}"
                )

            memory_block = (
                f"RELEVANT MEMORIES FROM PAST SESSIONS (retrieved by semantic similarity):\n"
                + "\n".join(memory_lines)
                + "\n\nUse these memories to personalise your response."
            )
            messages.append({"role": "system", "content": memory_block})
            # Injected as a system message — background context, not part of the dialogue.

        # 3. Recent buffer — current session turns verbatim.
        for msg in self._recent_buffer:
            messages.append(msg.to_api_format())
            # These provide conversational flow for the current session.

        return messages

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """Print current memory state — both ChromaDB and recent buffer."""
        user_count = self._get_user_memory_count()
        total_count = self._collection.count()

        print(f"\n{'='*62}")
        print(f"  Vector Store Stats — Session: {self.session_id}")
        print(f"{'='*62}")
        print(f"  Session turns        : {self._total_turns}")
        print(f"  Stored this session  : {self._stored_count} messages")
        print(f"  This user in ChromaDB: {user_count} messages (all sessions)")
        print(f"  Total in ChromaDB    : {total_count} messages (all users)")
        print(f"  Retrieved total      : {self._retrieved_count} memories")
        print(f"  Recent buffer size   : {len(self._recent_buffer)} messages")
        print(f"  Retrieval k          : {self.retrieval_k}")
        print(f"{'='*62}\n")


print("VectorStoreMemory class defined.")

VectorStoreMemory class defined.


---
## Part 4 — The FinCoach Agent

In [7]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation
  AND in the relevant memories from past sessions provided to you.
- Be specific with numbers when the user has provided their financial details.
- If you reference a memory from a past session, use it naturally without saying
  'according to my memory' — just incorporate it as if you remember.
- If you do not have information you need, ask for it rather than assuming.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions."""
# Key addition: instructions on how to use the retrieved memories naturally.
# Without this, some models will say things like "According to my memory bank..."
# which breaks the illusion of natural recall.


def chat(
    memory: VectorStoreMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """
    Send one user message to FinCoach with vector store memory.

    The flow:
    1. Retrieve relevant past memories from ChromaDB (READ)
    2. Build context: system + retrieved memories + recent buffer
    3. Call GPT-4o with this context
    4. Add user message and response to buffer + ChromaDB (WRITE)
    """

    # STEP 1 — Add the user message to buffer (but NOT to ChromaDB yet).
    # We add it to the buffer first so it appears in the recent context.
    # We store it in ChromaDB AFTER the API call to avoid retrieving
    # the current query as a 'past memory' (it would match itself perfectly).
    memory._recent_buffer.append(Message(role="user", content=user_message))
    if len(memory._recent_buffer) > memory.max_recent_turns * 2:
        memory._recent_buffer.pop(0)
    # Manual buffer management here because we are not calling add_message() yet.

    # STEP 2 — Build the message list: retrieve + buffer.
    api_messages = memory.get_messages_for_api(
        current_query=user_message,
        # Pass the current query so ChromaDB can find relevant past memories.
    )

    if verbose:
        retrieved_count = sum(
            1 for m in api_messages if m["role"] == "system" and "RELEVANT MEMORIES" in m["content"]
        )
        print(f"  [RETRIEVE] Query: '{user_message[:50]}...'" if len(user_message) > 50
              else f"  [RETRIEVE] Query: '{user_message}'")
        if retrieved_count:
            # Find the memory block and show the content.
            for m in api_messages:
                if m["role"] == "system" and "RELEVANT MEMORIES" in m["content"]:
                    lines = m["content"].split("\n")[1:-2]  # Strip header and footer.
                    for line in lines:
                        print(f"    ↳ {line[:80]}")
        else:
            print(f"  [RETRIEVE] No relevant past memories found")

    # STEP 3 — Call the API.
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=api_messages,
        # context = [system: persona] + [system: retrieved memories?] + [recent buffer]
    )

    assistant_reply = response.choices[0].message.content

    # STEP 4 — Now store the user message in ChromaDB.
    user_msg = memory._recent_buffer[-1]  # The user message we added in Step 1.
    memory._store_in_vector_db(user_msg)
    # Now it's in ChromaDB — retrievable in all future sessions.

    # STEP 5 — Add the assistant reply to buffer and optionally ChromaDB.
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        print(f"[Turn {memory._total_turns}] "
              f"API input: {response.usage.prompt_tokens} tokens | "
              f"ChromaDB: {memory._get_user_memory_count()} user memories")

    return assistant_reply


print("FinCoach chat() with vector store memory defined.")

FinCoach chat() with vector store memory defined.


---
## Part 5 — Demo: Session 1 — Building the Memory

Run Session 1 to populate ChromaDB with facts about Chiru. Every user message is embedded and stored. Session 2 will retrieve these facts without Chiru repeating himself.

In [8]:
# Session 1: Chiru shares his financial profile with FinCoach.
# These messages will be embedded and stored in ChromaDB.
# In a future session, FinCoach will remember them without being told again.

session1_memory = VectorStoreMemory(
    session_id="session_vs_001",
    user_id="chiru_001",
    # user_id scopes ALL ChromaDB queries — critical for multi-tenancy.
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    retrieval_k=3,
)

session1_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Embedded and stored → retrievable when future queries mention income.
    "My monthly expenses are ₹60,000 for rent, food, and transport.",
    # Stored → retrievable for savings and budget questions.
    "I have an FD of ₹50,000 maturing in 3 months. I'm conservative with investments.",
    # Stored → retrievable for investment and risk questions.
    "My long-term goal is to retire comfortably by age 55. I'm currently 32.",
    # Stored → retrievable for retirement planning questions.
    "Based on my profile, what should be my investment priority right now?",
    # Question turn — this message is also stored.
]

print("SESSION 1 — Building FinCoach memory for Chiru")
print("Every user message is embedded and stored in ChromaDB")
print("=" * 65)

for i, user_message in enumerate(session1_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=session1_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
session1_memory.print_stats()
print(f"ChromaDB now contains {session1_memory._get_user_memory_count()} memories for chiru_001")
print("These will be retrieved automatically in Session 2 — without Chiru repeating himself.")

[VectorStore] Initialised — session: session_vs_001
  User ID             : chiru_001
  Recent buffer turns : 5
  Retrieval k         : 3 messages per turn
  ChromaDB collection : 'fincoach_memory' (0 existing docs)
SESSION 1 — Building FinCoach memory for Chiru
Every user message is embedded and stored in ChromaDB

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
  [RETRIEVE] Query: 'Hi! I'm Chiru. My monthly take-home salary is ₹1,2...'
  [RETRIEVE] No relevant past memories found
[Turn 1] API input: 199 tokens | ChromaDB: 1 user memories
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, it's great that you're looking to manage your finances wisely. To start, it's important to create a budget that covers your essential expenses like rent, utilities, groceries, and insurance. Aim to allocate about 20% of your income, ₹24,000, to savings and investments. This could include an emergency fund, retirement savings, and any short-term goals you m

---
## Part 6 — Demo: Session 2 — Cross-Session Recall

A new session starts — the buffer is empty. But ChromaDB remembers everything from Session 1. Chiru asks questions that require his profile — without re-stating it.

In [9]:
# Session 2: fresh session — buffer empty, but ChromaDB intact.
# Chiru does NOT re-introduce himself. FinCoach should remember him.

session2_memory = VectorStoreMemory(
    session_id="session_vs_002",
    # NEW session ID — completely fresh buffer.
    user_id="chiru_001",
    # SAME user ID — ChromaDB will retrieve from Session 1's memories.
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=5,
    retrieval_k=3,
)

session2_turns = [
    "Hi, I'm back. Has my FD matured yet? What should I do with the money?",
    # Tests: does FinCoach remember the ₹50,000 FD from Session 1?
    # The query 'FD matured' should retrieve the FD message from ChromaDB.
    "Given my conservative risk profile, should I move it to a debt mutual fund?",
    # Tests: does FinCoach remember the risk profile without being told again?
    "How much monthly savings do I have available for new investments?",
    # Tests: does FinCoach remember salary (₹1,20,000) and expenses (₹60,000)?
    # Expected: FinCoach answers ₹60,000 surplus without being told.
    "Am I on track to retire by my target age?",
    # Tests: does FinCoach remember the retirement goal (age 55) and current age (32)?
]

print("SESSION 2 — Cross-Session Memory Recall")
print("Buffer is EMPTY — Chiru has not re-introduced himself")
print("FinCoach should retrieve relevant memories from Session 1 via ChromaDB")
print("=" * 65)

for i, user_message in enumerate(session2_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=session2_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
session2_memory.print_stats()

[VectorStore] Initialised — session: session_vs_002
  User ID             : chiru_001
  Recent buffer turns : 5
  Retrieval k         : 3 messages per turn
  ChromaDB collection : 'fincoach_memory' (5 existing docs)
SESSION 2 — Cross-Session Memory Recall
Buffer is EMPTY — Chiru has not re-introduced himself
FinCoach should retrieve relevant memories from Session 1 via ChromaDB

--- Turn 1 ---
User: Hi, I'm back. Has my FD matured yet? What should I do with the money?
  [RETRIEVE] Query: 'Hi, I'm back. Has my FD matured yet? What should I...'
    ↳ [2026-06-12] user: I have an FD of ₹50,000 maturing in 3 months. I'm conservativ
    ↳ [2026-06-12] user: Based on my profile, what should be my investment priority ri
[Turn 1] API input: 284 tokens | ChromaDB: 6 user memories
FinCoach: Welcome back! Since your FD of ₹50,000 was maturing in 3 months from June, it should have matured by now. Given your conservative investment approach, you might want to consider reinvesting it in another fixe

---
## Part 7 — The Stale Fact Problem

In [10]:
# This demo shows EXACTLY why pure vector store memory is insufficient
# for domains where facts change over time.

stale_test_memory = VectorStoreMemory(
    session_id="session_stale_test",
    user_id="chiru_stale_test",  # New user to avoid polluting chiru_001's memories.
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=3,
    retrieval_k=3,
)

# --- PHASE 1: Store an old fact ---
print("PHASE 1: Storing an old fact (6 months ago)")
old_fact = "I work at Infosys as a software engineer and earn ₹90,000 per month."
# Simulate this being from 6 months ago by using a past timestamp in the message.
old_message = Message(
    role="user",
    content=old_fact,
    timestamp="2024-12-01T10:00:00+00:00",  # 6 months ago.
    memory_id=str(uuid.uuid4())
)
stale_test_memory._store_in_vector_db(old_message)
print(f"  Stored: '{old_fact}'")

# --- PHASE 2: Store a new contradicting fact ---
print("\nPHASE 2: Storing a new contradicting fact (today)")
new_fact = "I recently changed jobs. I now work at TCS and my salary is ₹1,20,000 per month."
new_message = Message(
    role="user",
    content=new_fact,
    timestamp=datetime.now(timezone.utc).isoformat(),
    memory_id=str(uuid.uuid4())
)
stale_test_memory._store_in_vector_db(new_message)
print(f"  Stored: '{new_fact}'")

# --- PHASE 3: Query for both ---
print("\nPHASE 3: Query about employer and salary")
query = "Where do I work and what is my current salary?"

retrieved = stale_test_memory._retrieve_memories(
    query=query,
    exclude_current_session=False  # Include all memories including test data.
)

print(f"\nQuery: '{query}'")
print(f"Retrieved {len(retrieved)} memories:")
for i, mem in enumerate(retrieved, 1):
    ts = mem['metadata'].get('timestamp', '')[:10]
    dist = mem['distance']
    print(f"  {i}. [{ts}] distance={dist:.4f}: '{mem['content'][:70]}...'")

print()
print("THE STALE FACT PROBLEM:")
print("Both the old fact (Infosys, ₹90K) and the new fact (TCS, ₹1.2L) are retrieved.")
print("The vector store has no concept of 'which is current'.")
print("The model may blend them, pick the wrong one, or give a confused answer.")
print()
print("MITIGATION options implemented in this technique:")
print("  1. Filter by timestamp — only retrieve memories from the last N days")
print("  2. Similarity threshold — only inject very relevant memories")
print()
print("FULL SOLUTION → Technique 24 (Graphiti):")
print("  Temporal knowledge graphs track WHEN facts were true.")
print("  The old Infosys edge is marked invalid when TCS is added.")
print("  'Where do I work?' returns only the CURRENT employer.")

[VectorStore] Initialised — session: session_stale_test
  User ID             : chiru_stale_test
  Recent buffer turns : 3
  Retrieval k         : 3 messages per turn
  ChromaDB collection : 'fincoach_memory' (9 existing docs)
PHASE 1: Storing an old fact (6 months ago)
  Stored: 'I work at Infosys as a software engineer and earn ₹90,000 per month.'

PHASE 2: Storing a new contradicting fact (today)
  Stored: 'I recently changed jobs. I now work at TCS and my salary is ₹1,20,000 per month.'

PHASE 3: Query about employer and salary

Query: 'Where do I work and what is my current salary?'
Retrieved 2 memories:
  1. [2026-06-12] distance=0.4443: 'I recently changed jobs. I now work at TCS and my salary is ₹1,20,000 ...'
  2. [2024-12-01] distance=0.5211: 'I work at Infosys as a software engineer and earn ₹90,000 per month....'

THE STALE FACT PROBLEM:
Both the old fact (Infosys, ₹90K) and the new fact (TCS, ₹1.2L) are retrieved.
The vector store has no concept of 'which is current'.
The 

---
## Part 8 — Recency-Weighted Retrieval: Mitigation for the Stale Fact Problem

In [11]:
def retrieve_with_recency_boost(
    collection,
    user_id: str,
    query: str,
    k: int = 5,
    recency_weight: float = 0.3,
    days_halflife: int = 30,
) -> List[Dict]:
    """
    Retrieve memories with a recency boost applied to the similarity score.
    More recent memories score higher — partially mitigates the stale fact problem.

    Final score = semantic_similarity * (1 - recency_weight)
                + recency_score * recency_weight

    Args:
        collection:      The ChromaDB collection to query.
        user_id:         Filter to this user's memories only.
        query:           The search query text.
        k:               Number of memories to retrieve.
        recency_weight:  How much to weight recency vs semantic similarity (0-1).
                         0.0 = pure semantic similarity (no recency boost)
                         1.0 = pure recency (newest wins regardless of relevance)
                         0.3 = 30% recency, 70% semantic — a good production default
        days_halflife:   Recency score halves every this many days.
                         30 days: a message from 30 days ago scores 0.5 for recency.
                         A message from today scores 1.0.
    """

    import math

    # Retrieve more candidates than k — we will re-rank and return top k.
    candidates_k = min(k * 3, collection.count())
    if candidates_k == 0:
        return []

    query_embedding = embed_text(query)

    try:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=candidates_k,
            where={"user_id": {"$eq": user_id}},
            include=["documents", "metadatas", "distances"],
        )
    except Exception:
        return []

    if not results["documents"] or not results["documents"][0]:
        return []

    now = datetime.now(timezone.utc)
    scored = []

    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        # Semantic score: 1 - distance (higher = more similar).
        semantic_score = 1.0 - dist

        # Recency score: exponential decay from timestamp.
        try:
            msg_time = datetime.fromisoformat(meta["timestamp"])
            days_old = (now - msg_time).total_seconds() / 86400
            # days_old: how many days ago this message was created.
            recency_score = math.exp(-days_old / days_halflife)
            # Exponential decay: score = e^(-days_old / halflife)
            # Today (days_old=0):      score = 1.0
            # 30 days ago (halflife):  score = 0.37
            # 90 days ago (3x half):   score = 0.05
        except Exception:
            recency_score = 0.5  # Default if timestamp is missing or malformed.

        # Combined score: weighted blend of semantic and recency.
        combined_score = (
            semantic_score * (1 - recency_weight) +
            recency_score * recency_weight
        )

        scored.append({
            "content":        doc,
            "metadata":       meta,
            "semantic_score": round(semantic_score, 4),
            "recency_score":  round(recency_score, 4),
            "combined_score": round(combined_score, 4),
        })

    # Sort by combined score, highest first.
    scored.sort(key=lambda x: x["combined_score"], reverse=True)

    return scored[:k]  # Return only the top k after re-ranking.


# Test recency-weighted retrieval on the stale fact scenario.
print("RECENCY-WEIGHTED RETRIEVAL on the Infosys vs TCS stale fact")
print("=" * 65)

reranked = retrieve_with_recency_boost(
    collection=stale_test_memory._collection,
    user_id="chiru_stale_test",
    query="Where do I work and what is my salary?",
    k=3,
    recency_weight=0.3,
    days_halflife=30,
)

for i, mem in enumerate(reranked, 1):
    ts = mem['metadata'].get('timestamp', '')[:10]
    print(f"  {i}. [{ts}]")
    print(f"     Content  : '{mem['content'][:65]}...'")
    print(f"     Semantic : {mem['semantic_score']:.4f} | "
          f"Recency: {mem['recency_score']:.4f} | "
          f"Combined: {mem['combined_score']:.4f}")

print()
print("With recency weighting, the TCS/₹1.2L message (today) ranks higher")
print("than the Infosys/₹90K message (6 months ago) — even if semantic similarity")
print("is similar for both. Recency boost partially mitigates the stale fact problem.")
print("Partial mitigation only — the full solution is Graphiti (Technique 24).")

RECENCY-WEIGHTED RETRIEVAL on the Infosys vs TCS stale fact
  1. [2026-06-12]
     Content  : 'I recently changed jobs. I now work at TCS and my salary is ₹1,20...'
     Semantic : 0.5174 | Recency: 0.9999 | Combined: 0.6621
  2. [2024-12-01]
     Content  : 'I work at Infosys as a software engineer and earn ₹90,000 per mon...'
     Semantic : 0.5022 | Recency: 0.0000 | Combined: 0.3515

With recency weighting, the TCS/₹1.2L message (today) ranks higher
than the Infosys/₹90K message (6 months ago) — even if semantic similarity
is similar for both. Recency boost partially mitigates the stale fact problem.
Partial mitigation only — the full solution is Graphiti (Technique 24).


---
## Part 9 — Multi-Tenant Isolation: Verifying User Separation

In [12]:
# This test verifies that user_id filtering actually isolates memories.
# In production financial services, this is a compliance requirement.

print("MULTI-TENANT ISOLATION TEST")
print("=" * 65)

# Create a second user with different financial profile.
user2_memory = VectorStoreMemory(
    session_id="session_user2_001",
    user_id="priya_002",  # Different user entirely.
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_recent_turns=3,
    retrieval_k=3,
)

# Store a fact for user2.
user2_msg = Message(role="user", content="I am Priya. My salary is ₹80,000 per month.")
user2_memory._store_in_vector_db(user2_msg)
print(f"Stored for priya_002: '{user2_msg.content}'")

# Now query as chiru_001 for 'salary' — should NOT return Priya's salary.
print(f"\nQuerying as chiru_001 for 'salary' — should return Chiru's memories only")

chiru_retrieved = session1_memory._retrieve_memories(
    query="What is my salary?",
    exclude_current_session=False
)

print(f"Results for chiru_001 ({len(chiru_retrieved)} memories retrieved):")
for mem in chiru_retrieved:
    uid = mem['metadata']['user_id']
    print(f"  user_id={uid}: '{mem['content'][:60]}...'")

# Verify: none of the results should be from priya_002.
isolation_ok = all(m['metadata']['user_id'] == 'chiru_001' for m in chiru_retrieved)
print()
if isolation_ok:
    print("✅ ISOLATION VERIFIED: All retrieved memories belong to chiru_001 only.")
    print("   Priya's salary (₹80,000) was NOT returned when querying as Chiru.")
else:
    print("❌ ISOLATION FAILED: Cross-user data leak detected.")
    print("   Check the user_id filter in _retrieve_memories().")

print()
print("Production reminder: user_id in metadata is the ONLY thing preventing")
print("cross-user memory leakage. Test this explicitly in your CI/CD pipeline.")

MULTI-TENANT ISOLATION TEST
[VectorStore] Initialised — session: session_user2_001
  User ID             : priya_002
  Recent buffer turns : 3
  Retrieval k         : 3 messages per turn
  ChromaDB collection : 'fincoach_memory' (11 existing docs)
Stored for priya_002: 'I am Priya. My salary is ₹80,000 per month.'

Querying as chiru_001 for 'salary' — should return Chiru's memories only
Results for chiru_001 (3 memories retrieved):
  user_id=chiru_001: 'Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000....'
  user_id=chiru_001: 'My monthly expenses are ₹60,000 for rent, food, and transpor...'
  user_id=chiru_001: 'How much monthly savings do I have available for new investm...'

✅ ISOLATION VERIFIED: All retrieved memories belong to chiru_001 only.
   Priya's salary (₹80,000) was NOT returned when querying as Chiru.

Production reminder: user_id in metadata is the ONLY thing preventing
cross-user memory leakage. Test this explicitly in your CI/CD pipeline.


---
## Part 10 — The Architecture: Short-Term + Long-Term Together

In [13]:
# Illustrative — shows how the complete production architecture looks.
# This combines all techniques learned so far.

print("THE COMPLETE FINCOACH MEMORY ARCHITECTURE (so far)")
print("=" * 65)

print("""
EVERY API CALL RECEIVES:
─────────────────────────
[system: FinCoach persona]          ~130 tokens  (constant)
     +
[system: retrieved memories]         ~200 tokens  (variable — semantic search in ChromaDB)
     +
[recent buffer verbatim]            ~800 tokens  (bounded — last 5 turns)
─────────────────────────────────────────────────
TOTAL INPUT:                        ~1130 tokens  (predictable ceiling)


WRITE PIPELINE (after every turn):
───────────────────────────────────
1. User message → embed → store in ChromaDB with user_id + timestamp metadata
2. User message → add to recent buffer (evict oldest if buffer full)
3. Assistant reply → add to recent buffer only (not stored in ChromaDB by default)


READ PIPELINE (before every API call):
───────────────────────────────────────
1. Embed current user query
2. Query ChromaDB: WHERE user_id = 'chiru_001' ORDER BY cosine_similarity LIMIT 3
3. Filter: only include memories with distance < 0.7 (relevance threshold)
4. Inject retrieved memories as system context block
5. Append recent buffer turns


WHAT SURVIVES ACROSS SESSIONS:
────────────────────────────────
✅ ChromaDB: salary, expenses, risk profile, goals, FD, age — all retrievable
✅ Semantically: if you ask about 'income', it finds 'salary' messages
❌ Recent buffer: resets on new session — no conversational carry-over
❌ Temporal: cannot distinguish 'I used to work at Infosys' vs 'I work at TCS'

WHAT COMES NEXT:
─────────────────
Technique 07: Entity Memory  — structured facts (salary, risk, goals) in a profile store
Technique 08: Knowledge Graph — relationships between entities
Technique 24: Graphiti        — temporal knowledge graph: knows WHEN facts were true
""")

THE COMPLETE FINCOACH MEMORY ARCHITECTURE (so far)

EVERY API CALL RECEIVES:
─────────────────────────
[system: FinCoach persona]          ~130 tokens  (constant)
     +
[system: retrieved memories]         ~200 tokens  (variable — semantic search in ChromaDB)
     +
[recent buffer verbatim]            ~800 tokens  (bounded — last 5 turns)
─────────────────────────────────────────────────
TOTAL INPUT:                        ~1130 tokens  (predictable ceiling)


WRITE PIPELINE (after every turn):
───────────────────────────────────
1. User message → embed → store in ChromaDB with user_id + timestamp metadata
2. User message → add to recent buffer (evict oldest if buffer full)
3. Assistant reply → add to recent buffer only (not stored in ChromaDB by default)


READ PIPELINE (before every API call):
───────────────────────────────────────
1. Embed current user query
2. Query ChromaDB: WHERE user_id = 'chiru_001' ORDER BY cosine_similarity LIMIT 3
3. Filter: only include memories with dist

---
## Key Takeaways

**What you built:** A `VectorStoreMemory` class using ChromaDB and OpenAI embeddings for persistent, cross-session, semantically-retrieved long-term memory — with multi-tenant isolation, a similarity threshold, recency-weighted retrieval, and a stale fact demonstration.

---

### The three things to remember

1. **`user_id` in metadata is not optional — it is a compliance requirement.** Every ChromaDB `query()` call must include `where={"user_id": {"$eq": user_id}}`. Without it, your agent leaks one user's financial history to another. In financial services, this is a regulatory violation. Test this isolation explicitly in CI/CD.

2. **Vector stores retrieve by meaning, not by time.** A salary from three years ago is just as semantically similar to the query "what is my income" as a salary update from last week. Both will be retrieved. Both will be in the model's context. The model may use the wrong one. This is the stale fact problem — and it is the fundamental limitation that Graphiti (Technique 24) solves with bi-temporal knowledge graphs.

3. **Embed at write time, retrieve at read time.** The embedding call happens when the message is stored, not when it is retrieved. Retrieval is a vector similarity search — fast (sub-100ms for typical collections). The cost is at write time: one embedding API call per stored message. At $0.02 per million tokens, this is negligible for most production FinCoach deployments.

---

### What Technique 07 adds

Vector Store Memory retrieves by semantic similarity — great for fuzzy recall ("find messages about income"). But it returns raw message text — not structured facts. It cannot answer "what is Chiru's current salary" with a single structured value. **Technique 07 — Entity Memory** solves this: it extracts and maintains structured profile records (name: Chiru, salary: ₹1,20,000, risk: conservative) that are always current, always structured, and always injected — without semantic search.

---


### FAQ

Q: What is Vector Store Memory in agent memory?

A: Vector Store Memory embeds conversation turns or facts into high-dimensional vectors and stores them in a vector database (such as FAISS, Chroma, Pinecone, or Weaviate). On each turn, the system retrieves the top-K most semantically similar past entries rather than relying on recency. This enables relevance-based recall across thousands of stored memories. LangChain implements this as VectorStoreRetrieverMemory. Typical K values range from 3 to 10 entries per query.

Q: When should I use Vector Store Memory instead of Memory Retrieval Patterns?

A: Use basic Vector Store Memory when a single-stage semantic search meets your accuracy needs. Memory Retrieval Patterns (technique 20) adds BM25 keyword fusion, cross-encoder re-ranking, and MMR diversity filtering on top of vector search. Start with the simpler approach: embed and retrieve with cosine similarity. If you notice missed results due to keyword mismatches or redundant retrievals, upgrade to the multi-stage pipeline described in technique 20.

Q: What are the limits or failure modes of Vector Store Memory?

A: Embedding quality determines retrieval quality. If the embedding model misses domain-specific terms, relevant memories will not surface. Semantic search can return plausible but wrong matches (high similarity, low relevance). Write latency increases as the store grows, though query time remains sublinear with approximate nearest neighbor (ANN) indexes. There is also a cold-start problem: the system has nothing to retrieve until enough memories are stored.

Q: Can I combine Vector Store Memory with another memory technique?

A: Yes. A strong pattern pairs it with technique 02 (Sliding Window Memory) or technique 04 (Summary Buffer Memory). Keep the window or buffer for recent context, and query the vector store for relevant older memories. This gives you recency from the buffer and relevance from the vector store. You can also layer technique 18 (Temporal Memory) on top to add time-decay weighting to your similarity scores.

Q: What library or framework can I use to skip the implementation work?

A: LangChain provides VectorStoreRetrieverMemory with adapters for FAISS, Chroma, Pinecone, and Weaviate. LlamaIndex has native vector memory integration through its retriever modules. Mem0 (technique 25) uses vector storage under the hood with automatic embedding. Zep (technique 27) combines vector search with its knowledge graph for hybrid retrieval. For standalone vector stores, Chroma and FAISS are the most common open-source choices for prototyping.

